In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../'))

In [4]:
import warnings

import json
from numpy import random

from model.optimization import load_model, eval_model
from model.dataset import get_dataframe

import lightgbm as lgb

DEFAULT_RANDOM_SEED = 774
random.mtrand._rand.seed(DEFAULT_RANDOM_SEED)
warnings.filterwarnings("ignore")

category="min_tpm_5"

In [5]:
test_df = get_dataframe(category=category, dataset="test")
subtypes = test_df["subtype"].unique()

In [6]:
def run_tests(genes: int, subtype: str, report: bool = True):
  importances = json.loads(open(f"../../preprocessed/{category}/important_genes_random_forest_f1.json").readline())

  chosen_genes = list(set([y["gene"] for x in [sex_values[:genes] for sex_values in importances[subtype].values()] for y in x]))
  print(f"Total chosen genes: {len(chosen_genes)}")

  train_data = get_dataframe(category=category, dataset="train")
  train_data["subtype_target"] = train_data["subtype"] == subtype

  model = load_model(
    name="lgbm_feature_selection_random_forest_f1",
    model_factory=lambda **kwargs: lgb.LGBMClassifier(**kwargs, class_weight="balanced", verbosity=-1, n_jobs=-1, random_state=DEFAULT_RANDOM_SEED),
    dataset=(train_data[chosen_genes], train_data["subtype_target"])
  )
  
  test_data = test_df.copy()
  test_data["subtype_target"] = test_data["subtype"] == subtype
  return eval_model(model, dataset=(test_data[chosen_genes], test_data["subtype_target"]), report=report, use_threshold=True)
  

In [7]:
results = run_tests(genes=4, subtype="iAMP21")
print(results.report)

Total chosen genes: 8
Using threshold 0.014274837293231292, with max acc 0.9076305220883534
   f1_weighted  f1_macro  recall_macro  balanced_accuracy  accuracy
0     0.978335  0.851082      0.907631           0.907631  0.976744
              precision    recall  f1-score   support

       False       0.99      0.98      0.99       166
        True       0.62      0.83      0.71         6

    accuracy                           0.98       172
   macro avg       0.81      0.91      0.85       172
weighted avg       0.98      0.98      0.98       172



In [11]:
gene_count_by_subtype = {}
acc_by_subtype = {}
for subtype in subtypes:
  print(f"======== Subtype {subtype} ========")
  
  current_max = 0
  genes_max = 0
  for i in range(1, 10):
    acc = run_tests(genes=i, subtype=subtype, report=False).balanced_accuracy
    if acc > current_max + 0.01:
      current_max = acc
      genes_max = i

  gene_count_by_subtype[subtype] = genes_max
  acc_by_subtype[subtype] = current_max

======== Subtype HYPER ========
Total chosen genes: 2
Using threshold 0.003262128247947469, with max acc 0.8559964983950978
Total chosen genes: 4
Using threshold 0.19461295946585613, with max acc 0.8895535453749635
Total chosen genes: 5
Using threshold 1.5529128991343048e-05, with max acc 0.8927633498686898
Total chosen genes: 6
Using threshold 2.0145317592435086e-05, with max acc 0.8894076451707031
Total chosen genes: 8
Using threshold 0.005278075414548687, with max acc 0.8911584476218266
Total chosen genes: 10
Using threshold 0.00010524095966532811, with max acc 0.8894076451707031
Total chosen genes: 11
Using threshold 0.000639096398090551, with max acc 0.8978698570177999
Total chosen genes: 13
Using threshold 0.002061046982505739, with max acc 0.8727750218850306
Total chosen genes: 15
Using threshold 0.005238008774373761, with max acc 0.8761307265830172
======== Subtype PHlike ========
Total chosen genes: 2
Using threshold 0.00018421788834145094, with max acc 0.737233732127225
Total

In [12]:
gene_count_by_subtype

{'HYPER': 2,
 'PHlike': 7,
 'ETV6RUNX1': 2,
 'DUX4IGH': 1,
 'BCRABL1': 4,
 'HYPO': 4,
 'KMT2A': 5,
 'PAX5': 6,
 'TCF3PBX1': 2,
 'iAMP21': 4}

In [21]:
for subtype in acc_by_subtype:
    print(f"{subtype}: {'%.02f' % (acc_by_subtype[subtype] * 100)}%")

HYPER: 88.96%
PHlike: 90.46%
ETV6RUNX1: 98.56%
DUX4IGH: 99.67%
BCRABL1: 90.83%
HYPO: 84.08%
KMT2A: 99.67%
PAX5: 91.12%
TCF3PBX1: 100.00%
iAMP21: 90.76%


In [10]:
open(f"../../preprocessed/{category}/gene_count_by_subtype_random_forest_f1.json", "w").write(json.dumps(gene_count_by_subtype))

131

In [12]:
importances = json.loads(open(f"../../preprocessed/{category}/important_genes_random_forest_f1.json").readline())
for subtype in importances:
  print(f"===== {subtype} =====")
  for sex in importances[subtype]:
    print(f"===== {sex} =====")
    for gene in importances[subtype][sex][:gene_count_by_subtype[subtype]]:
      print(f"{gene['gene']} - {'%.04f' % gene['importance']}")

===== BCRABL1 =====
===== Male =====
WNT9A - 0.0116
PTPRO - 0.0078
TBXAS1 - 0.0068
IL2RA - 0.0065
===== Female =====
AL139415.1 - 0.0108
AL021878.2 - 0.0091
SPATS2L - 0.0076
FAR2P2 - 0.0058
===== DUX4IGH =====
===== Male =====
APELA - 0.0169
===== Female =====
APELA - 0.0131
===== ETV6RUNX1 =====
===== Male =====
ENSG00000286393 - 0.0197
DSC2 - 0.0130
===== Female =====
AP005530.1 - 0.0139
BIRC7 - 0.0136
===== PHlike =====
===== Male =====
AL139415.1 - 0.0120
NR3C2 - 0.0079
ENSG00000286848 - 0.0077
CCL17 - 0.0073
GLI2 - 0.0072
ABHD12B - 0.0068
TMEM136 - 0.0064
===== Female =====
TMEM236 - 0.0086
ENSG00000290021 - 0.0083
GLI2 - 0.0083
ENSG00000286848 - 0.0078
LDB3 - 0.0076
ENSG00000289050 - 0.0074
MYL9 - 0.0069
===== HYPER =====
===== Male =====
PAX5 - 0.0093
AL139275.1 - 0.0077
===== Female =====
LRFN2 - 0.0063
ENSG00000288101 - 0.0061
===== HYPO =====
===== Male =====
AP001057.1 - 0.0045
SLC9A3-AS1 - 0.0043
CTHRC1 - 0.0037
SCARNA22 - 0.0034
===== Female =====
HLA-DOA - 0.0150
HLA-DMB 